In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO
from collections import defaultdict

In [3]:
record = SeqIO.read("sequence.fasta", "fasta")
seq = str(record.seq)

In [4]:
print(f"Длина последовательности: {len(seq)} нуклеотидов")

Длина последовательности: 16299 нуклеотидов


Ну почти.. Но другой sequence.fasta не было в наличии

Подсчет всех динуклеотидов в последовательности

In [6]:
dinuc_counts = defaultdict(int)
nucleotides = ['A', 'C', 'G', 'T']

for i in range(len(seq) - 1):
    dinuc = seq[i:i+2]
    dinuc_counts[dinuc] += 1

total_dinucs = len(seq) - 1

print("Динуклеотид | Количество  | Частота")
for first in nucleotides:
    for second in nucleotides:
        dinuc = first + second
        count = dinuc_counts[dinuc]
        freq = count / total_dinucs
        print(f"     {dinuc}     |    {count:6d}   |  {freq:.6f}")


Динуклеотид | Количество  | Частота
     AA     |      1913   |  0.117376
     AC     |      1268   |  0.077801
     AG     |       796   |  0.048840
     AT     |      1651   |  0.101301
     CA     |      1365   |  0.083753
     CC     |      1104   |  0.067738
     CG     |       287   |  0.017610
     CT     |      1220   |  0.074856
     GA     |       662   |  0.040618
     GC     |       519   |  0.031844
     GG     |       397   |  0.024359
     GT     |       435   |  0.026690
     TA     |      1689   |  0.103632
     TC     |      1085   |  0.066573
     TG     |       532   |  0.032642
     TT     |      1375   |  0.084366


Построение матрицы переходов:

In [7]:
transition_matrix = np.zeros((4, 4))
nuc_to_idx = {nuc: i for i, nuc in enumerate(nucleotides)}

for i in range(len(seq) - 1):
    from_nuc = seq[i]
    to_nuc = seq[i+1]
    transition_matrix[nuc_to_idx[from_nuc], nuc_to_idx[to_nuc]] += 1

row_sums = transition_matrix.sum(axis=1)
for i in range(4):
    if row_sums[i] > 0:
        transition_matrix[i] = transition_matrix[i] / row_sums[i]

print("            A      C      G      T")
for i, from_nuc in enumerate(nucleotides):
    row = transition_matrix[i]
    print(f"  {from_nuc}     | ", end="")
    for j, prob in enumerate(row):
        print(f"{prob:.4f} ", end="")
    print()


            A      C      G      T
  A     | 0.3399 0.2253 0.1414 0.2934 
  C     | 0.3433 0.2777 0.0722 0.3068 
  G     | 0.3289 0.2578 0.1972 0.2161 
  T     | 0.3608 0.2318 0.1137 0.2937 


Проверка, что сумма элементов в каждой строке равна 1:

In [8]:
for i, from_nuc in enumerate(nucleotides):
    row_sum = transition_matrix[i].sum()
    print(f"Строка {from_nuc}: сумма = {row_sum:.6f}")

Строка A: сумма = 1.000000
Строка C: сумма = 1.000000
Строка G: сумма = 1.000000
Строка T: сумма = 1.000000


Нахождение стационарного распределения:

In [9]:
# Решаем уравнение pi = pi*P (pi*P = pi, pi * (P - I) = 0)
# Добавляем условие нормировки: сумма pi = 1

A = transition_matrix.T - np.eye(4)
A = np.vstack([A, np.ones(4)])
b = np.zeros(5)
b[-1] = 1

pi, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

for i, nuc in enumerate(nucleotides):
    print(f"  pi({nuc}) = {pi[i]:.6f}")
print(f"Сумма pi = {pi.sum():.6f}")


  pi(A) = 0.345381
  pi(C) = 0.243954
  pi(G) = 0.123447
  pi(T) = 0.287218
Сумма pi = 1.000000


Сравнение с наблюдаемыми частотами нуклеотидов:

In [10]:
observed_freqs = {}
for nuc in nucleotides:
    count = seq.count(nuc)
    observed_freqs[nuc] = count / len(seq)

In [13]:
print("Нуклеотид | Наблюдаемая частота")
for i, nuc in enumerate(nucleotides):
    obs = observed_freqs[nuc]
    stat = pi[i]
    diff = abs(obs - stat)
    print(f"    pi({nuc}) = {obs:.6f}      ")


Нуклеотид | Наблюдаемая частота
    pi(A) = 0.345359      
    pi(C) = 0.243941      
    pi(G) = 0.123505      
    pi(T) = 0.287196      


Видим, что наблюдаемые частоты совпадают со стационарным распределением до 4 цифры после запятой, дальше начинаются расхождения